# Staggered DiD Failure Lab: Why Naive TWFE Can Lie

**Econometrics Notebook Library · v0.1.0**

## Intuition

With staggered treatment timing, a two-way fixed-effects event study can compare already-treated units to newly-treated units. If treatment effects differ across cohorts or evolve with exposure length, those are not clean counterfactual comparisons. The coefficient on event time $k$ can therefore mix the target effect at $k$ with treatment effects from other relative periods.

This notebook makes the failure visible before introducing the modern estimators.

**Target:** dynamic average treatment effects on treated observations by event time.

## Algebraic warning

The familiar regression

$$
Y_{it}=\alpha_i+\lambda_t+\sum_{k\neq -1}\beta_k 1\{t-G_i=k\}+\varepsilon_{it}
$$

looks like a sequence of causal contrasts. Under heterogeneous effects, however, Frisch–Waugh–Lovell residualization gives each $\beta_k$ an implicit set of comparison weights. Those weights need not isolate $ATT_k$ and can be negative or load on other relative times.

Sun & Abraham show that apparent pre-trends can be generated *solely* by heterogeneous post-treatment effects. Borusyak–Jaravel–Spiess show that conventional regressions can fail absent strong treatment-effect restrictions.

References: [Sun & Abraham (2021)](https://doi.org/10.1016/j.jeconom.2020.09.006) · [Borusyak, Jaravel & Spiess (2024)](https://doi.org/10.1093/restud/rdae007)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_staggered_panel, twfe_event_study

df = simulate_staggered_panel(n_units=260, seed=21)
twfe = twfe_event_study(df, window=(-3, 4))
truth = (df[df.treated.eq(1)]
         .groupby("event_time", as_index=False)["tau_true"].mean()
         .rename(columns={"tau_true":"truth"}))
out = twfe.merge(truth, on="event_time", how="left")
out

In [ ]:
fig, ax = plt.subplots()
ax.axhline(0, linewidth=1)
ax.plot(out.event_time, out.truth, marker="o", label="True ATT by event time")
ax.errorbar(out.event_time, out.estimate, yerr=1.96*out.se, marker="o", capsize=3, label="Naive TWFE")
ax.set(xlabel="Event time", ylabel="Effect", title="Same data, different object")
ax.legend();

## What failed?

The data-generating process satisfies parallel trends for untreated potential outcomes. The problem is not a hidden pre-trend. It is the estimator's comparison structure under staggered adoption plus treatment-effect heterogeneity.

A modern workflow therefore asks two separate questions:

1. **Identification:** which untreated observations are valid controls for cohort $g$ at time $t$?
2. **Aggregation:** which cohort-time effects do we actually want to average, and with what weights?

The next three notebooks answer those questions differently.

## Researcher failure checklist

- Do not interpret a TWFE lead coefficient as a pure pre-trend diagnostic under heterogeneous dynamic effects.
- Do not let already-treated units silently serve as controls unless the estimand and assumptions justify it.
- Report the treatment-timing support: which cohorts identify each horizon?
- State the aggregation weights. “Event-study coefficient” is not an estimand.
- Compare at least one heterogeneity-robust estimator when adoption is staggered.